# 25b — one-Gaussian (bandwidth-free) linewidth uncertainty

Based on 25a: the KDE Fisher's γ uncertainty swung ~1000× with the bandwidth because its
score carries a 1/h² kernel derivative. Here we drop the KDE for γ and assume the fitted
FWHM is normally distributed across scans (CLT for a fitted statistic), so

    σ_γ = σ_FWHM / (∂FWHM/∂γ · √N)   —  scatter ÷ sensitivity ÷ √N, no bandwidth.

μ's score (n−μ)/σ_prop² is already bandwidth-free (no h term), so μ is shown via the
KDE@Scott against the direct-count baseline σ_prop/(μ√N). We also check the normality
assumption (skew/kurtosis of the FWHM cloud).

Result: gamma — Gaussian sigma_gamma ~ KDE@Scott x1 at high T (both 0.1-0.5% for T>=40, skew +0.5..+1.7), diverging at low T where normality fails (skew up to +6.6): 1nW T5 = KDE 110% vs Gaussian 9.7%. mu — already bandwidth-free; KDE@Scott blows up at low N (94% at 1nW T5 vs the count baseline 3.5%, falling to ~1% for T>=40), i.e. the (FWHM,sigma_fit) summary discards most of the count information. Verdict: Gaussian is a valid bandwidth-free gamma replacement wherever the summary is Gaussian (T>=40); at low T it is non-Gaussian and mu is information-starved.

In [1]:
import os, sys, time
for _up in ('', '..', '../..'):
    _p = os.path.abspath(_up)
    if os.path.exists(os.path.join(_p, 'src', 'fisher.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break

import numpy as np
import torch

from src.fisher import (EXPERIMENTS, bandwidths, fisher_at, fisher_gaussian,
                        load_real_target, make_pool, sim_cloud)

M_FINAL = 500
SEED = 42
N_WORKERS = os.cpu_count() or 4
torch.set_num_threads(1)
print('ready —', len(EXPERIMENTS), 'experiments,', N_WORKERS, 'workers')

ready — 14 experiments, 4 workers


In [2]:
results = []
t0 = time.time()
with make_pool(N_WORKERS) as pool:
    for (power, trans, mu_true, sigma_prop, lam, gamma_true) in EXPERIMENTS:
        target_f, target_s = load_real_target(int(power.replace('nW', '')), trans)
        cloud = sim_cloud(mu_true, sigma_prop, lam, gamma_true, M_FINAL, SEED, pool=pool)
        h_f, h_s = bandwidths(target_f, target_s, 1.0, 1.0, 0.05)   # Scott x1
        kde = fisher_at(cloud, target_f, target_s, h_f, h_s, mu_true, sigma_prop,
                        gamma_scale=False, sigma_weight=1.0)
        gauss = fisher_gaussian(cloud, len(target_f))
        results.append(dict(power=power, trans=trans, N=len(target_f), gamma_true=gamma_true,
                            mu_true=mu_true, sigma_prop=sigma_prop,
                            rel_mu_kde=kde['sig_mu_data'] / mu_true,
                            rel_mu_count=sigma_prop / (mu_true * len(target_f) ** 0.5),
                            rel_g_kde=kde['sig_g_data'] / gamma_true,
                            rel_g_gauss=gauss['sig_g_data'] / gamma_true,
                            sigma_fwhm=gauss['sigma_fwhm'], j=gauss['j'],
                            skew=gauss['skew'], kurt=gauss['kurt']))
        print(f"[{power} T{trans}] N={len(target_f)}  mu={100*kde['sig_mu_data']/mu_true:.1f}%  "
              f"g_kde={100*kde['sig_g_data']/gamma_true:.1f}%  g_gauss={100*gauss['sig_g_data']/gamma_true:.1f}%  "
              f"skew={gauss['skew']:+.2f}  ({time.time()-t0:.0f}s)")

print(f"done in {time.time()-t0:.0f}s — {len(results)} rows")

[1nW T5] N=61  mu=94.0%  g_kde=109.9%  g_gauss=9.7%  skew=+2.19  (10s)


[1nW T10] N=358  mu=24.9%  g_kde=3.6%  g_gauss=4.6%  skew=+6.57  (19s)


[1nW T20] N=1138  mu=2.0%  g_kde=0.5%  g_gauss=1.8%  skew=+4.28  (27s)


[1nW T40] N=2428  mu=1.0%  g_kde=0.3%  g_gauss=0.5%  skew=+0.84  (36s)


[1nW T60] N=2424  mu=1.2%  g_kde=0.2%  g_gauss=0.4%  skew=+0.60  (44s)


[1nW T80] N=2487  mu=1.2%  g_kde=0.2%  g_gauss=0.3%  skew=+0.55  (53s)


... (4 outputs left out)

[3nW T40] N=3742  mu=0.7%  g_kde=0.1%  g_gauss=0.3%  skew=+0.95  (96s)


[3nW T60] N=2541  mu=0.9%  g_kde=0.1%  g_gauss=0.3%  skew=+0.73  (104s)


[3nW T80] N=2508  mu=0.9%  g_kde=0.1%  g_gauss=0.3%  skew=+0.51  (113s)


[3nW T100] N=2516  mu=1.3%  g_kde=0.1%  g_gauss=0.2%  skew=+0.52  (122s)
done in 122s — 14 rows


In [3]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

trans_order = [5, 10, 20, 40, 60, 80, 100]
gcol, kcol, ccol = 'crimson', 'steelblue', 'grey'

# Fig 1: mu (amplitude) + gamma (linewidth) uncertainty, one figure per power
for power in ('1nW', '3nW'):
    rows = sorted([r for r in results if r['power'] == power],
                  key=lambda r: trans_order.index(r['trans']))
    xs = [r['trans'] for r in rows]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('μ amplitude (relative)', 'γ linewidth (relative)'),
                        vertical_spacing=0.15)
    fig.add_trace(go.Scatter(x=xs, y=[100*r['rel_mu_kde'] for r in rows], mode='lines+markers',
                             name='μ KDE@Scott', line=dict(color=kcol, width=3),
                             marker=dict(size=8)), row=1, col=1)
    fig.add_trace(go.Scatter(x=xs, y=[100*r['rel_mu_count'] for r in rows], mode='lines+markers',
                             name='μ count (σ_prop/μ√N)', line=dict(color=ccol, width=2, dash='dot'),
                             marker=dict(size=6)), row=1, col=1)
    fig.add_trace(go.Scatter(x=xs, y=[100*r['rel_g_gauss'] for r in rows], mode='lines+markers',
                             name='γ Gaussian (no bandwidth)', line=dict(color=gcol, width=3),
                             marker=dict(size=8)), row=2, col=1)
    fig.add_trace(go.Scatter(x=xs, y=[100*r['rel_g_kde'] for r in rows], mode='lines+markers',
                             name='γ KDE@Scott', line=dict(color=kcol, width=3, dash='dash'),
                             marker=dict(size=8)), row=2, col=1)
    fig.add_hline(y=10.0, line_dash='dot', line_color='lightgrey', row=1, col=1)
    fig.add_hline(y=10.0, line_dash='dot', line_color='lightgrey', row=2, col=1)
    fig.update_layout(title=f'Fisher uncertainty at truth — {power}',
                      height=700, hovermode='x unified', xaxis2_title='transmission (%)')
    fig.update_yaxes(title_text='σ / truth (%)', row=1, col=1)
    fig.update_yaxes(title_text='σ / truth (%)', row=2, col=1)
    display({'application/vnd.plotly.v1+json': pio.to_json(fig)}, raw=True)

# Fig 2: normality check — skew and excess kurtosis of the FWHM cloud
fig2 = make_subplots(rows=1, cols=2, subplot_titles=('skewness', 'excess kurtosis'),
                     shared_xaxes=True)
for power, col in (('1nW', 1), ('3nW', 2)):
    rows = sorted([r for r in results if r['power'] == power],
                  key=lambda r: trans_order.index(r['trans']))
    xs = [r['trans'] for r in rows]
    fig2.add_trace(go.Bar(x=xs, y=[r['skew'] for r in rows], name=power,
                          marker_color=gcol if power == '1nW' else kcol,
                          showlegend=(col == 1)), row=1, col=1)
    fig2.add_trace(go.Bar(x=xs, y=[r['kurt'] for r in rows], name=power,
                          marker_color=gcol if power == '1nW' else kcol,
                          showlegend=False), row=1, col=2)
fig2.add_hline(y=0.0, line_color='grey', line_dash='dot', row=1, col=1)
fig2.add_hline(y=0.0, line_color='grey', line_dash='dot', row=1, col=2)
fig2.update_layout(title='Normality of the simulated FWHM (0 = Gaussian)',
                   height=400, xaxis2_title='transmission (%)')
display({'application/vnd.plotly.v1+json': pio.to_json(fig2)}, raw=True)